In [2]:
import pandas as pd
import numpy as np
import random
from faker import Faker

# Initialize Faker instance for generating synthetic feedback
fake = Faker()

# Function to generate random user feedback
def generate_feedback():
    feedbacks = [
        "I love the smart spoon! The taste enhancement is amazing.",
        "It's good, but not as effective as expected.",
        "Not happy with the device, it doesn't change the taste much.",
        "The taste enhancement is great, but the device is a bit bulky.",
        "I don't like the taste enhancement, I feel it's inconsistent.",
        "This is the best device for enhancing taste perception!",
        "It works, but I wish the effect lasted longer."
    ]
    return random.choice(feedbacks)

# Generate synthetic data
num_samples = 200

data = {
    'feedback': [generate_feedback() for _ in range(num_samples)],
    'satisfaction_level': [random.randint(1, 5) for _ in range(num_samples)],  # Rating 1-5
    'usage_data': [random.randint(1, 10) for _ in range(num_samples)],  # Number of times used per week
}

# Create a DataFrame
df = pd.DataFrame(data)

# Function to perform sentiment analysis (positive/negative)
def sentiment_analysis(feedback):
    positive_keywords = ['love', 'amazing', 'best', 'great']
    negative_keywords = ['not', 'wish', 'inconsistent', 'disappointed']

    # Simple sentiment classification based on keywords
    if any(word in feedback.lower() for word in positive_keywords):
        return 1  # Positive sentiment
    elif any(word in feedback.lower() for word in negative_keywords):
        return 0  # Negative sentiment
    return random.choice([0, 1])  # Default to random if unclear

# Apply sentiment analysis to the feedback
df['sentiment'] = df['feedback'].apply(sentiment_analysis)

# Show the synthetic dataset
print(df.head())

# Save to CSV for further use
df.to_csv("synthetic_user_feedback.csv", index=False)


                                            feedback  satisfaction_level  \
0     It works, but I wish the effect lasted longer.                   5   
1  I love the smart spoon! The taste enhancement ...                   3   
2  Not happy with the device, it doesn't change t...                   1   
3  I don't like the taste enhancement, I feel it'...                   4   
4     It works, but I wish the effect lasted longer.                   4   

   usage_data  sentiment  
0           5          0  
1          10          1  
2           5          0  
3           9          0  
4           9          0  


In [ ]:
import pandas as pd
import numpy as np
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:
df = pd.read_csv("user_feedback.csv")
print(df.head())
print(df.info())

In [ ]:
def preprocess_feedback(text):
    tokens = word_tokenize(text.lower())
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words and word.isalpha()]
    
    return " ".join(filtered_tokens)

df['cleaned_feedback'] = df['feedback'].apply(preprocess_feedback)

In [ ]:
def get_sentiment(text):
    blob = TextBlob(text)
    return 1 if blob.sentiment.polarity > 0 else 0

df['sentiment'] = df['cleaned_feedback'].apply(get_sentiment)

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)
X_feedback = vectorizer.fit_transform(df['cleaned_feedback']).toarray()

In [ ]:
X_user_behavior = df[['satisfaction_level', 'usage_data']].values
X = np.hstack((X_feedback, X_user_behavior))
y = df['sentiment'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Machine Learning Model 1: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

In [ ]:
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf)}")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Machine Learning Model 2: Support Vector Classifier (SVC)
svc_model = SVC(kernel='linear', random_state=42)
svc_model.fit(X_train, y_train)
y_pred_svc = svc_model.predict(X_test)
print(f"SVC Accuracy: {accuracy_score(y_test, y_pred_svc)}")
print(classification_report(y_test, y_pred_svc))

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_svc)

plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix for Sentiment Prediction')
plt.show()

In [ ]:
def predict_improvements(user_data):
    user_feedback = vectorizer.transform([user_data['feedback']]).toarray()
    user_behavior = np.array([user_data['satisfaction_level'], user_data['usage_data']]).reshape(1, -1)
    
    user_features = np.hstack((user_feedback, user_behavior))
    
    sentiment_pred = rf_model.predict(user_features)
    
    if sentiment_pred == 1:
        improvement = "No improvement needed, user is satisfied."
    else:
        improvement = "Improvement needed, user is dissatisfied."
    
    return improvement

In [ ]:
user_data = {
    'feedback': "The spoon is great, but the taste enhancement is not noticeable.",
    'satisfaction_level': 3,
    'usage_data': 5 
}
improvement = predict_improvements(user_data)
print(improvement)